# 돌봄부담 데이터, 무엇이 들어 있나

발달장애인 보호자 **3,000가구**의 실태조사 응답을 살펴본다.
모델을 만들기 전에 데이터가 어떻게 생겼는지 눈으로 확인하는 단계다.

---

## 이 노트북이 답하는 질문 열 가지

| | 질문 | 왜 궁금한가 |
|---|---|---|
| 1 | 데이터가 얼마나 있나 | 학습에 충분한 양인지 |
| 2 | 맞혀야 할 답은 어떻게 생겼나 | 어느 쪽이 많고 적은지 |
| 3 | 연습용과 시험용을 공정하게 나눴나 | 나누기가 틀리면 점수도 틀린다 |
| 4 | 빈칸은 왜 생겼나 | 안 물어본 것과 답을 안 한 것은 다르다 |
| 5 | **빈칸에 함정이 있나** | 이 데이터의 가장 큰 발견 |
| 6 | 어떤 질문이 답을 잘 맞히나 | 설문을 줄일 근거 |
| 7 | 실제로 어떻게 갈리나 | 숫자가 아니라 그림으로 확인 |
| 8 | 서로 겹치는 질문이 있나 | 같은 걸 두 번 묻고 있지 않은지 |
| 9 | 진짜 중요한 질문은 뭔가 | 겹치는 걸 걷어낸 뒤 |
| 10 | 어디로 신청하러 가나 | 지도 기능이 쓸 데이터 |

---

> ### 먼저 알아둘 것 하나
>
> 돌봄부담은 **1~5 단계**로 되어 있는데, **숫자가 작을수록 부담이 큽니다.**
>
> | 1 | 2 | 3 | 4 | 5 |
> |---|---|---|---|---|
> | 최고부담 | 고부담 | 중간 | 저부담 | 부담 없음 |
>
> 헷갈리기 쉬워서, 아래 모든 그래프에는 숫자 대신 **이름**으로 표시합니다.

## 준비

처음 한 번만 설치하면 된다.

In [ ]:
# 처음 한 번만 — 앞의 # 을 지우고 실행
# %pip install pandas numpy matplotlib seaborn scikit-learn scipy sqlalchemy pymysql

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sqlalchemy import create_engine, text

# ── 한글이 깨지지 않게 폰트 지정 ─────────────────────────────
for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"]:
    if _f in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams["font.family"] = _f
        print("한글 폰트:", _f)
        break
else:
    print("한글 폰트가 없습니다. 그래프 글자가 □ 로 보이면 NanumGothic 을 설치하세요.")

# ── 발표용 기본 설정 (글자 크게, 군더더기 없이) ───────────────
plt.rcParams.update({
    "axes.unicode_minus": False,
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
})

# ── 부담 단계 색: 진할수록 부담이 크다 ───────────────────────
색 = ["#3b1f3f", "#6f3566", "#a45178", "#cd8583", "#e6bfa4"]
이름 = {1: "최고부담", 2: "고부담", 3: "중간", 4: "저부담", 5: "부담 없음"}
강조색 = "#8a4a2a"

pd.set_option("display.max_rows", 100)
print("준비 완료")

## 데이터 가져오기

팀 데이터베이스에 접속해서 세 가지를 가져온다.

| 가져오는 것 | 내용 |
|---|---|
| `cb_dataset_v1` | 설문 응답 3,000건 (학습에 쓸 데이터) |
| `cb_feature_meta_v1` | 각 질문의 종류·빈칸 개수 정리표 |
| `services_with_coords_std09` | 복지서비스 제공기관 위치 |

In [ ]:
# 팀 데이터베이스 접속 정보
접속 = {
    "host":     "mis.iptime.org",
    "port":     13306,
    "user":     "pioneer3",
    "password": "pioneer26",
    "database": "ABC8pioneer3",
}

engine = create_engine(
    "mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4".format(**접속),
    pool_pre_ping=True,
)

with engine.connect() as con:
    print("접속 성공 —", con.execute(text("SELECT VERSION()")).scalar())
    print("데이터베이스 —", con.execute(text("SELECT DATABASE()")).scalar())

In [ ]:
df   = pd.read_sql("SELECT * FROM cb_dataset_v1", engine)          # 설문 응답
meta = pd.read_sql("SELECT * FROM cb_feature_meta_v1", engine)     # 질문 정리표
svc  = pd.read_sql("SELECT * FROM services_with_coords_std09", engine)  # 기관 위치

질문들 = meta["feature"].tolist()
print(f"설문 응답    {len(df):>6,} 건")
print(f"질문 개수    {len(질문들):>6} 개")
print(f"기관 목록    {len(svc):>6,} 곳")

In [ ]:
# 컬럼 이름이 영어라 그래프에 쓸 한글 이름을 붙여둔다
한글 = {
 "help_needed_hours":"일상생활 도움이 얼마나 필요한가",
 "understands_work_meaning":"일의 의미를 이해하는 정도",
 "can_work_standard_job":"일반 회사에서 일할 수 있는 정도",
 "caregiver_age":"보호자 나이",
 "age_disability_suspected":"장애를 처음 의심한 나이",
 "severe_dd_job_willingness":"중증인 경우 일할 의향",
 "family_support_for_employment":"가족이 취업을 지지하는 정도",
 "daily_routine_satisfaction":"일상생활 만족도",
 "overall_health":"전반적인 건강 상태",
 "wants_person_employed":"보호자가 취업을 바라는지",
 "past_employment_exp":"과거에 일한 적 있는지",
 "employment_status":"어떤 형태로 일하는지",
 "job_ability_mobility":"이동 능력",
 "is_employed":"지금 일하고 있는지",
 "school_helpfulness":"학교 교육이 도움이 된 정도",
 "job_ability_strength":"근력",
 "household_head_type":"생계를 책임지는 사람",
 "final_school":"최종 학력",
 "has_chronic_disease":"만성질환 종류",
 "relation_to_person":"보호자와 당사자의 관계",
 "last_job_quit_reason":"마지막 직장을 그만둔 이유",
 "primary_caregiver_type":"주로 돌보는 사람",
 "household_size":"가구원 수",
 "health_change_yoy":"작년보다 건강이 어떤지",
 "has_job_skill_cert":"자격증이 있는지",
 "lives_with_mother":"어머니와 같이 사는지",
 "past_job_count":"과거 직장 수",
 "lives_with_father":"아버지와 같이 사는지",
 "secondary_caregiver_type":"보조로 돌보는 사람",
 "lives_with_person":"당사자와 같이 사는지",
 "person_marital_status":"당사자 혼인 상태",
 "completion_status":"학교를 마쳤는지",
 "caregiver_gender":"보호자 성별",
 "caregiver_employment_status":"보호자가 일하는지",
 "wanted_to_stay_at_last_job":"그 직장을 계속 다니고 싶었는지",
 "early_aging":"또래보다 빨리 노화하는지",
 "exercises_regularly":"규칙적으로 운동하는지",
 "person_gender":"당사자 성별",
}
KO = lambda c: 한글.get(c, c)

# 앞으로 계속 쓸 조각들
연습용 = df[df.split == "train"].copy()    # 모델을 가르칠 때 쓰는 부분
시험용 = df[df.split == "test"].copy()     # 마지막에 딱 한 번 채점할 때 쓰는 부분
척도 = meta.set_index("feature")["var_scale"]     # continuous / ordinal / binary / nominal
숫자질문 = [c for c in 질문들 if 척도[c] == "continuous"]   # 나이·개수처럼 크기가 있는 것
선택질문 = [c for c in 질문들 if 척도[c] != "continuous"]   # 보기에서 고르는 것

빠진것 = [c for c in 질문들 if c not in 한글]
if 빠진것:
    print("주의 — 한글 이름이 없는 컬럼:", 빠진것)
    for c in 빠진것:
        한글[c] = c

print(f"연습용 {len(연습용):,}건  ·  시험용 {len(시험용):,}건\n")
print("질문 종류")
_설명 = {"continuous":"숫자로 답함 (나이·개수)", "ordinal":"순서가 있는 보기 (1~5 등)",
        "binary":"예/아니오", "nominal":"순서 없는 보기 (종류·관계)"}
for k, n in 척도.value_counts().items():
    print(f"  {k:<12} {n:>2}개   {_설명[k]}")


In [ ]:
# 각 질문의 보기가 무슨 뜻인지, 숫자가 커지면 어느 쪽인지
설명표 = meta[["feature","var_scale","n_levels","missing_pct","scale_note"]].copy()
설명표.insert(0, "질문", 설명표["feature"].map(KO))
설명표 = 설명표.rename(columns={"var_scale":"종류","n_levels":"보기 수",
                              "missing_pct":"빈칸%","scale_note":"보기 설명"})
display(설명표.drop(columns="feature").sort_values("종류").reset_index(drop=True))

---

# 1. 맞혀야 할 답은 어떻게 생겼나

우리가 예측하려는 것은 보호자의 **돌봄부담 단계**다.
1(최고부담)부터 5(부담 없음)까지 다섯 단계로 되어 있다.

**무엇을 보나** — 다섯 단계에 사람이 고르게 있는지, 한쪽으로 쏠렸는지
**왜 보나** — 한쪽이 너무 적으면 모델이 그쪽을 못 배운다

In [ ]:
개수 = df["care_burden"].value_counts().sort_index()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios":[1.5, 1]})

# 왼쪽 — 단계별 사람 수
ax[0].bar([이름[i] for i in 개수.index], 개수.values, color=색, width=.6)
for x, v in enumerate(개수.values):
    ax[0].text(x, v + 25, f"{v:,}명\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10)
ax[0].set_ylim(0, 개수.max()*1.25)
ax[0].set_ylabel("가구 수")
ax[0].set_title("부담이 큰 쪽에 사람이 더 많다", loc="left")

# 오른쪽 — 도움이 급한 쪽 vs 아닌 쪽
급함 = df["care_burden"].isin([1, 2]).sum()
여유 = len(df) - 급함
ax[1].barh(["도움이 급한 쪽\n(최고부담 + 고부담)", "그 외\n(중간 이하)"], [급함, 여유],
           color=[색[1], 색[3]], height=.5)
for y, v in enumerate([급함, 여유]):
    ax[1].text(v + 30, y, f"{v:,}명 ({v/len(df)*100:.0f}%)", va="center", fontsize=10.5)
ax[1].set_xlim(0, max(급함, 여유) * 1.4)
ax[1].invert_yaxis()
ax[1].grid(axis="y", visible=False)
ax[1].set_title("절반 이상이 도움이 필요한 쪽", loc="left")

plt.tight_layout(); plt.show()

print(f"가장 많은 단계는 '{이름[개수.idxmax()]}' {개수.max():,}명")
print(f"가장 적은 단계는 '{이름[개수.idxmin()]}' {개수.min():,}명")
print(f"→ 가장 많은 쪽이 가장 적은 쪽의 {개수.max()/개수.min():.0f}배")

### 여기서 알 수 있는 것

숫자가 한쪽으로 쏠려 있다. 보통 이런 걸 **불균형**이라고 하고 문제로 본다.

그런데 **이 데이터는 쏠린 방향이 우리에게 유리하다.**

- 우리가 찾아야 하는 사람 = 부담이 큰 보호자 → **1,629명, 전체의 54%**
- 가장 적은 쪽 = "부담 없음" → 77명뿐이지만, 이 사람들은 **애초에 서비스가 필요 없다**

찾아야 할 사람이 많으니 모델이 배우기 쉽다.
따로 데이터를 부풀리는 기법(오버샘플링)은 쓸 필요가 없다.

---

# 2. 연습용과 시험용을 공정하게 나눴나

3,000명을 두 무리로 나눠 놓았다.

- **연습용 2,398명** — 모델을 가르칠 때 쓴다
- **시험용 602명** — 다 끝난 뒤 딱 한 번 채점할 때만 쓴다

**왜 확인하나** — 시험용에 유독 특정한 사람만 몰려 있으면, 나중에 점수가 이상하게 나와도
그건 모델 잘못이 아니라 **나누기를 잘못한 탓**이다. 먼저 확인하고 넘어간다.

**어떻게 읽나** — 두 무리의 비율 차이가 **1%p 안쪽이면 잘 나뉜 것**이다.

In [ ]:
비교 = pd.DataFrame({
    "연습용 %": 연습용["care_burden"].value_counts(normalize=True).sort_index() * 100,
    "시험용 %": 시험용["care_burden"].value_counts(normalize=True).sort_index() * 100,
})
비교["차이"] = 비교["시험용 %"] - 비교["연습용 %"]
비교.index = [이름[i] for i in 비교.index]
display(비교.round(1))

fig, ax = plt.subplots(figsize=(9, 4))
x, w = np.arange(5), .36
ax.bar(x - w/2, 비교["연습용 %"], w, label=f"연습용 {len(연습용):,}명", color=색[1])
ax.bar(x + w/2, 비교["시험용 %"], w, label=f"시험용 {len(시험용):,}명", color=색[3])
for i in x:
    ax.text(i, max(비교["연습용 %"].iloc[i], 비교["시험용 %"].iloc[i]) + 1.2,
            f"차이 {비교['차이'].iloc[i]:+.1f}%p", ha="center", fontsize=9, color="#666")
ax.set_xticks(x); ax.set_xticklabels(비교.index)
ax.set_ylabel("비율 (%)"); ax.set_ylim(0, 비교[["연습용 %","시험용 %"]].values.max()*1.25)
ax.legend(frameon=False)
ax.set_title("두 무리의 구성이 거의 같다", loc="left")
plt.tight_layout(); plt.show()

최대차 = 비교["차이"].abs().max()
print(f"가장 큰 차이: {최대차:.2f}%p")
print("→ 잘 나뉘었습니다. 그대로 써도 됩니다." if 최대차 < 1
      else "→ 차이가 1%p 를 넘습니다. 분할을 다시 확인하세요.")

---

# 3. 빈칸은 왜 생겼나

응답에 빈칸이 꽤 있다. 그런데 **빈칸에는 두 종류**가 있다.

| 종류 | 예 | 어떻게 다룰까 |
|---|---|---|
| **안 물어본 것** | 일한 적이 없는 사람에게 "왜 그만뒀나요?" 를 물을 이유가 없다 | 이건 정보다. "해당 없음" 으로 남겨야 한다 |
| **답을 안 한 것** | 물어봤는데 모른다고 한 경우 | 이건 진짜 빈칸이다. 채워 넣어야 한다 |

둘을 똑같이 다루면 안 된다. **일한 적이 없는 사람에게 없는 퇴사 이유를 지어내게 되기 때문**이다.

**무엇을 보나** — 어느 질문이 얼마나 비어 있는지

In [ ]:
빈칸 = (df[질문들].isna().mean() * 100).sort_values(ascending=False)
빈칸 = 빈칸[빈칸 > 0]

fig, ax = plt.subplots(figsize=(9.5, len(빈칸)*0.42 + 1))
ax.barh([KO(c) for c in 빈칸.index], 빈칸.values, color=색[2], height=.6)
for y, v in enumerate(빈칸.values):
    ax.text(v + 1.2, y, f"{v:.0f}%", va="center", fontsize=10)
ax.invert_yaxis(); ax.set_xlim(0, 112)
ax.set_xlabel("비어 있는 비율 (%)")
ax.grid(axis="y", visible=False)
ax.set_title(f"질문 {len(질문들)}개 중 {len(빈칸)}개에 빈칸이 있다", loc="left")
plt.tight_layout(); plt.show()

print(f"빈칸이 전혀 없는 질문: {len(질문들) - len(빈칸)}개")
print(f"절반 넘게 비어 있는 질문: {(빈칸 >= 50).sum()}개")

---

# 4. 빈칸의 함정 — 이 데이터의 가장 큰 발견

앞에서 "안 물어봐서 빈칸" 이라고 했다. 그런데 여기에 함정이 있다.

**"안 물어봤다"는 사실이 이미 다른 질문의 답에 적혀 있다.**

```
"과거에 일한 적 있나요?"  →  아니오
        ↓
"직장을 몇 곳 다녔나요?"        → 물어볼 필요 없음 → 빈칸
"왜 그만뒀나요?"                → 물어볼 필요 없음 → 빈칸
"계속 다니고 싶었나요?"          → 물어볼 필요 없음 → 빈칸
```

세 개가 비어 있다는 것은 **"일한 적 없음"이라는 답을 세 번 더 말하는 것**과 같다.

**확인 방법** — 앞 질문의 답만 보고 뒤 질문이 비었는지 맞혀본다. 하나도 안 틀리면 완전히 겹치는 것이다.

In [ ]:
def 확인(뒤질문, 조건식, 설명):
    비었나 = df[뒤질문].isna()
    예상   = df.eval(조건식)
    틀린수 = int((비었나 != 예상).sum())
    return {"뒤 질문": KO(뒤질문), "앞 질문의 답": 설명,
            "빈칸 수": int(비었나.sum()), "틀린 개수": 틀린수,
            "겹치는 비율": f"{(1 - 틀린수/len(df))*100:.2f}%"}

규칙 = [
 ("past_job_count",             "past_employment_exp.isna() or past_employment_exp==2", "일한 적 없음"),
 ("last_job_quit_reason",       "past_employment_exp.isna() or past_employment_exp==2", "일한 적 없음"),
 ("wanted_to_stay_at_last_job", "past_employment_exp.isna() or past_employment_exp==2", "일한 적 없음"),
 ("employment_status",          "is_employed==2", "지금 일하지 않음"),
 ("past_employment_exp",        "is_employed==1", "지금 일하고 있음"),
 ("wants_person_employed",      "is_employed==1", "지금 일하고 있음"),
]
결과 = pd.DataFrame([확인(a, b, c) for a, b, c in 규칙])
display(결과)

완전 = (결과["틀린 개수"] == 0).sum()
거의 = (결과["틀린 개수"] <= 3).sum()
print(f"{len(규칙)}쌍 중 {완전}쌍은 하나도 안 틀렸고, {거의}쌍은 3건 이하로 틀렸습니다.")
print("→ 빈칸이 앞 질문의 답만으로 거의 완벽하게 예측됩니다. 새로운 정보가 없다는 뜻입니다.\n")
if 결과["틀린 개수"].max() > 0:
    print("몇 건 틀리는 이유")
    print("  '안 물어봐서 빈칸' 과 '물어봤는데 답을 안 해서 빈칸' 이 둘 다 NULL 로 들어 있습니다.")
    print("  뒤쪽은 앞 질문으로 예측되지 않으므로 그만큼 틀립니다. 3,000건 중 한두 건 수준입니다.")

In [ ]:
# 빈칸이 생기는 패턴이 똑같은 질문끼리 묶어보기
패턴 = df[빈칸.index].isna()
묶음 = {}
for c in 패턴.columns:
    묶음.setdefault(tuple(패턴[c].values), []).append(c)

같은묶음 = [v for v in 묶음.values() if len(v) > 1]
print("빈칸이 완전히 똑같이 생기는 질문 묶음\n")
for cols in 같은묶음:
    print(f"  [{패턴[cols[0]].sum():,}건이 빈칸]")
    for c in cols:
        print(f"      · {KO(c)}")
    print()

fig, ax = plt.subplots(figsize=(9, max(len(빈칸)*0.32, 3)))
정렬 = 패턴[빈칸.index].astype(int)
정렬 = 정렬.sort_values(list(정렬.columns), ascending=False)
ax.imshow(정렬.T.values, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(len(빈칸)))
ax.set_yticklabels([KO(c) for c in 빈칸.index], fontsize=9)
ax.set_xlabel("응답자 3,000명 (빈칸 패턴이 비슷한 순서로 정렬)")
ax.set_xticks([]); ax.grid(False)
ax.set_title("검은 부분이 빈칸 — 세로줄이 나란한 질문은 서로 겹친다", loc="left")
plt.tight_layout(); plt.show()

### 여기서 알 수 있는 것

**여섯 쌍 모두 하나도 안 틀렸다.** 앞 질문의 답만 알면 뒤 질문이 비었는지 완벽히 맞힌다.

이게 왜 문제인가.

모델은 "이 칸이 비었네" 라는 사실만으로도 답을 추측한다.
그런데 그 정보는 이미 앞 질문에 있으므로, **같은 이야기를 네 번 듣는 셈**이 된다.

그러면 이런 일이 생긴다.

| | 결과 |
|---|---|
| 성적 | 크게 나빠지지 않는다. 중복이지 거짓말은 아니니까 |
| **질문 고르기** | **중요도가 네 갈래로 쪼개져서, 정작 중요한 질문이 밀려난다** |
| **결과 설명** | 보호자에게 *"부담이 큰 이유: 과거 직장 수"* 라고 표시된다. 실제 이유는 "일한 적이 없어서" 인데도 |

이 프로젝트의 목표가 **질문 줄이기**와 **이유 설명하기** 이므로, 그냥 넘길 수 없다.

---

# 5. 어떤 질문이 답을 잘 맞히나

질문 하나만 보고 부담 단계를 얼마나 알 수 있는지 재본다.

**설명력** 이라고 부르고, **%로 표시**한다.

- 설명력 8% = 이 질문 하나로 답의 8% 정도를 알 수 있다
- 설명력 0% = 이 질문은 답과 아무 상관이 없다

> 연습용 2,398명만 써서 계산한다. 시험용을 섞으면 나중 채점이 후해진다.

In [ ]:
from sklearn.metrics import mutual_info_score
from scipy.stats import entropy

정답 = 연습용["care_burden"]
전체불확실성 = entropy(정답.value_counts(normalize=True), base=2)

def 설명력(질문):
    답 = 연습용[질문].fillna("빈칸").astype(str)
    return mutual_info_score(답, 정답) / np.log(2) / 전체불확실성 * 100

순위 = pd.DataFrame({
    "질문": [KO(c) for c in 질문들],
    "설명력%": [round(설명력(c), 2) for c in 질문들],
    "컬럼": 질문들,
}).sort_values("설명력%", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.5, len(순위)*0.33 + 1))
d = 순위.iloc[::-1]
색깔 = [색[0] if v >= 5 else 색[1] if v >= 2 else 색[2] if v >= 1 else 색[3] for v in d["설명력%"]]
ax.barh(d["질문"], d["설명력%"], color=색깔, height=.66)
for y, v in enumerate(d["설명력%"]):
    ax.text(v + .12, y, f"{v:.1f}", va="center", fontsize=9)
ax.set_xlabel("설명력 (%)  —  이 질문 하나로 답을 얼마나 알 수 있나")
ax.grid(axis="y", visible=False)
ax.set_title("질문마다 힘이 다르다", loc="left")
plt.tight_layout(); plt.show()

print("가장 힘센 질문 5개")
for _, r in 순위.head(5).iterrows():
    print(f"  {r['설명력%']:>5.1f}%   {r['질문']}")
print(f"\n설명력 1% 도 안 되는 질문: {(순위['설명력%'] < 1).sum()}개")
print("→ 이 질문들은 빼도 될 가능성이 있습니다. (확정은 모델로 확인)")

### 잠깐 — 가장 힘센 질문은 빼버렸다

원래 데이터에는 **"보호자 생활만족도"** 라는 질문이 있었고, 설명력이 **19%** 로 압도적 1위였다.
그런데 이 질문은 쓰지 않기로 했다.

> 생활만족도로 돌봄부담을 맞히는 것은 **"힘드세요?" 라고 물어서 "힘든가 봅니다" 라고 답하는 것**과 같다.
> 점수는 잘 나오지만 진단 도구로서는 아무 의미가 없다.

같은 이유로 여섯 개를 뺐다. 얼마나 손해인지 확인해 본다.

In [ ]:
raw = pd.read_sql("SELECT * FROM `2024_care_burden_std09`", engine)

뺀질문 = {
 "caregiver_life_satisfaction": "보호자 생활만족도",
 "care_difficulty_top1":        "돌볼 때 가장 어려운 점",
 "needed_care_service_type":    "필요한 돌봄서비스 종류",
 "work_care_gap_hours":         "돌봄 때문에 쉰 기간",
 "work_care_gap_exp":           "돌봄 때문에 일을 쉰 적",
 "integrated_care_awareness":   "통합돌봄을 아는지",
}
정답r = raw["care_burden"]
불확실r = entropy(정답r.value_counts(normalize=True), base=2)
뺀것 = pd.Series({
    v: mutual_info_score(raw[k].fillna("빈칸").astype(str), 정답r) / np.log(2) / 불확실r * 100
    for k, v in 뺀질문.items()
}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.barh(뺀것.index, 뺀것.values, color=강조색, height=.55, label="뺀 질문")
ax.barh(순위["질문"].head(3), 순위["설명력%"].head(3), color=색[2], height=.55, label="남은 질문 중 상위 3")
for y, v in enumerate(뺀것.values):
    ax.text(v + .3, y, f"{v:.1f}%", va="center", fontsize=10)
ax.invert_yaxis(); ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
ax.set_xlabel("설명력 (%)")
ax.set_title("뺀 질문이 남은 질문보다 힘이 셌다", loc="left")
plt.tight_layout(); plt.show()

print(f"뺀 것 중 최고  {뺀것.max():.1f}%")
print(f"남은 것 중 최고 {순위['설명력%'].max():.1f}%")
print("→ 절반 넘게 잃었지만, 이걸 쓰면 진단 도구가 아니라 동어반복이 됩니다.")

---

# 6. 실제로 어떻게 갈리나

숫자만 보면 감이 안 온다. 힘센 질문 여섯 개가 **답을 실제로 어떻게 가르는지** 그림으로 본다.

**어떻게 읽나** — 막대 하나가 그 답을 고른 사람들이다.
**왼쪽 진한 색이 길수록 부담이 큰 사람이 많다는 뜻**이다.

In [ ]:
보기 = 순위[순위["컬럼"].map(lambda c: meta.set_index("feature").loc[c, "n_levels"]) <= 6].head(6)

fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, (_, r) in zip(axes.ravel(), 보기.iterrows()):
    c = r["컬럼"]
    표 = pd.crosstab(연습용[c].fillna("빈칸"), 연습용["care_burden"], normalize="index") * 100
    표 = 표.reindex(columns=[1,2,3,4,5], fill_value=0)
    왼쪽 = np.zeros(len(표))
    for j, 단계 in enumerate([1,2,3,4,5]):
        ax.barh(표.index.astype(str), 표[단계], left=왼쪽, color=색[j], height=.66,
                label=이름[단계] if ax is axes[0,0] else None)
        왼쪽 += 표[단계].values
    인원 = 연습용[c].fillna("빈칸").value_counts()
    ax.set_yticklabels([f"{i}  ({인원[i]:,}명)" for i in 표.index], fontsize=9.5)
    ax.set_xlim(0, 100); ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_title(f"{r['질문']}\n설명력 {r['설명력%']:.1f}%", loc="left", fontsize=11)
    ax.set_xlabel("비율 (%)", fontsize=9.5)
fig.legend(loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(.5, -.02), fontsize=11)
plt.tight_layout(); plt.show()

---

# 7. 서로 겹치는 질문이 있나

앞에서 빈칸이 겹치는 걸 봤다. 이번에는 **답 자체가 겹치는지** 본다.

두 질문이 얼마나 붙어 다니는지를 **0에서 1 사이 숫자**로 잰다.

```
0     전혀 상관 없다
0.5   꽤 비슷하게 움직인다
1     완전히 같은 질문이다
```

**0.5 를 넘으면 둘 중 하나는 빼도 되는지 따져봐야 한다.**

In [ ]:
from scipy.stats import chi2_contingency

def 겹침정도(a, b):
    """두 질문이 얼마나 붙어 다니는지 0~1 로 반환"""
    표 = pd.crosstab(a.fillna("빈칸").astype(str), b.fillna("빈칸").astype(str))
    if min(표.shape) < 2:
        return 0.0
    카이 = chi2_contingency(표, correction=False)[0]
    n = 표.values.sum()
    return float(np.sqrt((카이 / n) / max(min(표.shape) - 1, 1)))

상위 = 순위["컬럼"].head(12).tolist()
M = pd.DataFrame(index=상위, columns=상위, dtype=float)
for i, a in enumerate(상위):
    for j, b in enumerate(상위):
        M.iloc[i, j] = 1.0 if i == j else (M.iloc[j, i] if j < i else 겹침정도(연습용[a], 연습용[b]))

fig, ax = plt.subplots(figsize=(9.5, 8))
가림 = np.triu(np.ones_like(M, dtype=bool), k=1)
sns.heatmap(M.astype(float), mask=가림, cmap="rocket_r", vmin=0, vmax=1, square=True,
            linewidths=.7, linecolor="white", annot=True, fmt=".2f", annot_kws={"size":9},
            xticklabels=[KO(c) for c in 상위], yticklabels=[KO(c) for c in 상위],
            cbar_kws={"label": "겹치는 정도 (0~1)", "shrink": .6}, ax=ax)
ax.set_title("힘센 질문 12개끼리 얼마나 겹치나", loc="left", pad=12)
plt.setp(ax.get_xticklabels(), rotation=40, ha="right", fontsize=9)
plt.setp(ax.get_yticklabels(), fontsize=9)
ax.grid(False)
plt.tight_layout(); plt.show()

쌍 = [(a, b, M.loc[a, b]) for i, a in enumerate(상위) for b in 상위[i+1:] if M.loc[a, b] >= .45]
print("많이 겹치는 질문 쌍 (0.45 이상)")
for a, b, v in sorted(쌍, key=lambda x: -x[2]):
    print(f"  {v:.2f}   {KO(a)}   ↔   {KO(b)}")
if not 쌍:
    print("  없음 — 상위 질문끼리는 서로 다른 이야기를 하고 있습니다.")

---

# 8. 진짜 중요한 질문은 뭔가

지금까지 잰 **설명력**은 질문을 **하나씩 따로** 봤을 때의 힘이다.
겹치는 질문들은 각자 점수를 받는다.

이번에는 **다른 질문을 전부 놓고, 이 질문만 없앴을 때 성적이 얼마나 떨어지는지** 잰다.
겹치는 질문은 없애도 성적이 안 떨어진다. 다른 질문이 대신 말해주기 때문이다.

```
설명력  =  이 질문 하나만 보면 얼마나 아나
영향력  =  다른 질문이 다 있는데도 이 질문이 필요한가
```

> 여기서 만드는 모델은 **비교용 기준선**이다. 실제 모델 만들기는 다음 단계다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score

# 보기에서 고르는 질문 → 빈칸은 "해당 없음(-1)" 으로 두고 보기별로 칸을 나눔
# 숫자로 답하는 질문 → 빈칸은 중앙값으로 채우고 크기를 맞춤
준비 = ColumnTransformer([
    ("고르는질문", Pipeline([("빈칸", SimpleImputer(strategy="constant", fill_value=-1)),
                          ("펼치기", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), 선택질문),
    ("숫자질문",   Pipeline([("빈칸", SimpleImputer(strategy="median")),
                          ("크기맞추기", StandardScaler())]), 숫자질문),
])
모델 = Pipeline([("준비", 준비),
               ("분류", LogisticRegression(max_iter=2000, class_weight="balanced"))])

X, y = 연습용[질문들], 연습용["care_burden"]
모델.fit(X, y)
성적 = f1_score(y, 모델.predict(X), average="macro")
print(f"기준선 모델 성적 (연습용 기준): {성적:.3f}")
print("※ 이 숫자는 참고용입니다. 진짜 채점은 모든 결정이 끝난 뒤 시험용으로 한 번만 합니다.\n")

측정 = permutation_importance(모델, X, y, n_repeats=10, random_state=0,
                            scoring="f1_macro", n_jobs=-1)
영향력 = (pd.DataFrame({"컬럼": 질문들, "영향력": 측정.importances_mean})
           .assign(질문=lambda d: d["컬럼"].map(KO))
           .sort_values("영향력", ascending=False).reset_index(drop=True))
display(영향력.head(10)[["질문", "영향력"]].round(4))

In [ ]:
비교표 = (영향력.set_index("컬럼")[["영향력"]]
          .join(순위.set_index("컬럼")[["설명력%", "질문"]]))
비교표["설명력순위"] = 비교표["설명력%"].rank(ascending=False)
비교표["영향력순위"] = 비교표["영향력"].rank(ascending=False)
비교표["순위변화"] = (비교표["설명력순위"] - 비교표["영향력순위"]).round(0)

fig, ax = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={"width_ratios":[1.2, 1]})

d = 영향력.head(15).iloc[::-1]
ax[0].barh(d["질문"], d["영향력"], color=색[1], height=.66)
ax[0].axvline(0, color="#888", lw=1)
ax[0].grid(axis="y", visible=False)
ax[0].set_xlabel("이 질문을 빼면 성적이 얼마나 떨어지나")
ax[0].set_title("다른 질문이 다 있어도 꼭 필요한 질문", loc="left")

ax[1].scatter(비교표["설명력순위"], 비교표["영향력순위"], s=42, color=색[2],
              edgecolors="white", linewidths=.7)
끝 = len(비교표) + 1
ax[1].plot([0, 끝], [0, 끝], color="#999", ls="--", lw=1)
큰변화 = 비교표.reindex(비교표["순위변화"].abs().sort_values(ascending=False).index).head(5)
for _, r in 큰변화.iterrows():
    ax[1].annotate(r["질문"], (r["설명력순위"], r["영향력순위"]), fontsize=9,
                   xytext=(6, 5), textcoords="offset points", color=강조색)
ax[1].set_xlabel("혼자 봤을 때 순위"); ax[1].set_ylabel("다 놓고 봤을 때 순위")
ax[1].invert_xaxis(); ax[1].invert_yaxis()
ax[1].set_title("점이 선에서 멀수록 순위가 크게 바뀐 질문", loc="left")
plt.tight_layout(); plt.show()

print("순위가 많이 바뀐 질문")
for _, r in 큰변화.iterrows():
    변화 = int(r["순위변화"])
    설명 = "혼자 볼 땐 세 보였지만 실은 겹침" if 변화 < 0 else "다 놓고 봐도 살아남음"
    print(f"  {변화:+3d}위   {r['질문']:<32} {설명}")

---

# 9. 어디로 신청하러 가나

진단이 끝나면 가까운 기관을 안내해야 한다. 그 기관 목록이 어떻게 생겼는지 본다.

**무엇을 보나** — 몇 곳이 있고, 전국에 고르게 있는지, 좌표가 제대로 들어 있는지

In [ ]:
print("서비스 종류")
print(svc["사업유형"].value_counts().to_string())

문제 = {
    "좌표 없음":      int(svc[["lat","lng"]].isna().any(axis=1).sum()),
    "좌표가 한국 밖": int((~svc.lat.between(33, 39.5) | ~svc.lng.between(124, 132)).sum()),
    "전화번호 없음":  int(svc["전화번호"].isna().sum()),
}
print("\n데이터 상태")
for k, v in 문제.items():
    print(f"  {k:<14} {v:>4}건" + ("   ← 확인 필요" if v else "   OK"))

print("\n시군구 커버리지 (전국 229개 기준)")
for 종류, 수 in svc.groupby("사업유형")["시군구"].nunique().items():
    print(f"  {종류:<24} {수:>3}개  ({수/229*100:.0f}%)")
전체 = svc["시군구"].nunique()
print(f"  {'합쳐서':<24} {전체:>3}개  ({전체/229*100:.0f}%)")

In [ ]:
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=.25)

ax0 = fig.add_subplot(gs[0])
for 종류, c, m in zip(svc["사업유형"].unique(), [색[1], 색[3]], ["o", "^"]):
    s = svc[svc["사업유형"] == 종류]
    ax0.scatter(s.lng, s.lat, s=13, c=c, marker=m, alpha=.72,
                edgecolors="white", linewidths=.3, label=f"{종류} {len(s):,}곳")
ax0.set_aspect(1/np.cos(np.radians(36)))
ax0.set_xlabel("경도"); ax0.set_ylabel("위도")
ax0.legend(loc="lower left", frameon=False, fontsize=9)
ax0.set_title("전국에 퍼져 있고 좌표는 모두 정상", loc="left")

ax1 = fig.add_subplot(gs[1])
표 = svc.pivot_table(index="시도", columns="사업유형", values="제공기관_명",
                    aggfunc="count", fill_value=0)
표 = 표.loc[표.sum(axis=1).sort_values().index]
왼쪽 = np.zeros(len(표))
for c, col in zip([색[1], 색[3]], 표.columns):
    ax1.barh(표.index, 표[col], left=왼쪽, color=c, height=.7, label=col)
    왼쪽 += 표[col].values
for y, v in enumerate(표.sum(axis=1)):
    ax1.text(v + 5, y, f"{v:,}", va="center", fontsize=9)
ax1.grid(axis="y", visible=False); ax1.set_xlabel("기관 수")
ax1.legend(loc="lower right", frameon=False, fontsize=9)
ax1.set_title("경기도에 가장 많다", loc="left")
plt.show()

---

# 정리 — 학습 전에 정할 것 네 가지

| | 무엇을 봤나 | 그래서 무엇을 할까 |
|---|---|---|
| **1** | 부담이 큰 쪽이 절반이 넘는다 (54%) | 데이터를 부풀리는 기법은 필요 없다 |
| **2** | 빈칸 여섯 쌍이 앞 질문과 100% 겹친다 | 겹치는 질문을 정리하지 않으면 **중요도와 이유 설명이 흐려진다** |
| **3** | 거의 다 비어 있는 질문이 세 개 있다 (88%) | 빼는 것을 우선 검토. 설문이 세 문항 줄어든다 |
| **4** | 설명력 1% 미만이 절반 가까이 된다 | 질문을 줄일 여지가 실제로 있다 |

### 빈칸을 다루는 원칙

```
안 물어본 빈칸   →   "해당 없음" 이라는 하나의 답으로 취급
                     (채워 넣으면 없는 사실을 지어내게 된다)

답을 안 한 빈칸   →   채워 넣는다
                     단, 채울 값은 연습용에서만 계산한다
```

### 하지 말아야 할 것

- **PCA 로 질문을 압축하기** — 압축해도 물어볼 질문 수는 그대로다.
  질문 38개를 성분 5개로 줄여도, 성분을 계산하려면 38개를 다 물어야 한다.
  우리에게 필요한 것은 **압축이 아니라 골라내기**다.
- **성적이 안 나온다고 뺐던 질문을 되살리기** — 생활만족도를 넣으면 점수는 오르지만
  진단 도구로서는 무의미해진다.